# Python Insights - Analisando Dados com Python

### Case - Cancelamento de Clientes

Você foi contratado por uma empresa com mais de 800 mil clientes para um projeto de Dados. Recentemente a empresa percebeu que da sua base total de clientes, a maioria são clientes inativos, ou seja, que já cancelaram o serviço.

Precisando melhorar seus resultados ela quer conseguir entender os principais motivos desses cancelamentos e quais as ações mais eficientes para reduzir esse número.

Base de dados e arquivos: https://drive.google.com/drive/folders/1uDesZePdkhiraJmiyeZ-w5tfc8XsNYFZ?usp=drive_link

In [2]:
# importar a base de dados

import pandas as pd

tabela = pd.read_csv("cancelamentos_sample.csv")
# removendo coluna customerid da base
tabela = tabela.drop(columns="CustomerID")

print(tabela)
print(tabela.info())

       idade    sexo  tempo_como_cliente  frequencia_uso  ligacoes_callcenter  \
0       23.0    Male                13.0            22.0                  2.0   
1       49.0    Male                55.0            16.0                  3.0   
2       30.0    Male                 7.0             1.0                  0.0   
3       26.0    Male                40.0             5.0                  3.0   
4       27.0  Female                17.0            30.0                  5.0   
...      ...     ...                 ...             ...                  ...   
49995   62.0  Female                35.0             7.0                  2.0   
49996   36.0    Male                43.0            21.0                  2.0   
49997   55.0    Male                42.0             8.0                  1.0   
49998   40.0  Female                14.0            19.0                  1.0   
49999   64.0    Male                41.0            29.0                  5.0   

       dias_atraso assinatu

In [16]:
# removendo linhas vazias da base
tabela = tabela.dropna()
print(tabela.info())

<class 'pandas.core.frame.DataFrame'>
Index: 49996 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   idade                   49996 non-null  float64
 1   sexo                    49996 non-null  object 
 2   tempo_como_cliente      49996 non-null  float64
 3   frequencia_uso          49996 non-null  float64
 4   ligacoes_callcenter     49996 non-null  float64
 5   dias_atraso             49996 non-null  float64
 6   assinatura              49996 non-null  object 
 7   duracao_contrato        49996 non-null  object 
 8   total_gasto             49996 non-null  float64
 9   meses_ultima_interacao  49996 non-null  float64
 10  cancelou                49996 non-null  float64
dtypes: float64(8), object(3)
memory usage: 4.6+ MB
None


In [25]:
# analisar percentual de cancelamento

print(tabela["cancelou"].value_counts())
print(tabela["cancelou"].value_counts(normalize=True))
print(tabela["cancelou"].value_counts(normalize=True).map("{:.1%}".format))

cancelou
1.0    28393
0.0    21603
Name: count, dtype: int64
cancelou
1.0    0.567905
0.0    0.432095
Name: proportion, dtype: float64
cancelou
1.0    56.8%
0.0    43.2%
Name: proportion, dtype: object


In [27]:
# analisar percentual de duracao do contrato


print(tabela["duracao_contrato"].value_counts())
print("####################################################################")
print(tabela["duracao_contrato"].value_counts(normalize=True))
print("####################################################################")
print(tabela["duracao_contrato"].value_counts(normalize=True).map("{:.1%}".format))

duracao_contrato
Annual       20156
Quarterly    19956
Monthly       9884
Name: count, dtype: int64
####################################################################
duracao_contrato
Annual       0.403152
Quarterly    0.399152
Monthly      0.197696
Name: proportion, dtype: float64
####################################################################
duracao_contrato
Annual       40.3%
Quarterly    39.9%
Monthly      19.8%
Name: proportion, dtype: object


In [28]:
# analisar tabela por duracao de contrato

analise_por_contrato = tabela.groupby("duracao_contrato").mean(numeric_only=True)
print(analise_por_contrato)

                      idade  tempo_como_cliente  frequencia_uso   
duracao_contrato                                                  
Annual            38.783985           31.416452       15.910449  \
Monthly           41.428470           30.677964       15.577600   
Quarterly         38.833935           31.522099       15.842504   

                  ligacoes_callcenter  dias_atraso  total_gasto   
duracao_contrato                                                  
Annual                       3.277585    12.533985   650.925840  \
Monthly                      4.923917    15.078814   547.508921   
Quarterly                    3.256113    12.480006   654.102443   

                  meses_ultima_interacao  cancelou  
duracao_contrato                                    
Annual                         14.231544  0.464080  
Monthly                        15.392655  1.000000  
Quarterly                      14.364602  0.458759  


In [29]:
# todos os clientes com o contrato mensal cancelaram.
# fazer analise so cancelamento sem o contrato mensal

tabela = tabela[tabela["duracao_contrato"] != "Monthly"]
print(tabela["cancelou"].value_counts())
print(tabela["cancelou"].value_counts(normalize=True))
print(tabela["cancelou"].value_counts(normalize=True).map("{:.1%}".format))

cancelou
0.0    21603
1.0    18509
Name: count, dtype: int64
cancelou
0.0    0.538567
1.0    0.461433
Name: proportion, dtype: float64
cancelou
0.0    53.9%
1.0    46.1%
Name: proportion, dtype: object


In [36]:
# analisar graficos

import plotly.express as px

for i in tabela.columns:
    grafico = px.histogram(tabela, x = i, color="cancelou")
    grafico.show()

# teste
## teste
### teste
#### teste
##### teste
###### teste

In [37]:
# com os graficos a gente consegue descobrir muita coisa:
# dias atraso acima de 20 dias, 100% cancela
# ligações call center acima de 5 todo mundo cancela

tabela = tabela[tabela["ligacoes_callcenter"]<5]
tabela = tabela[tabela["dias_atraso"]<=20]
display(tabela)
display(tabela["cancelou"].value_counts())
display(tabela["cancelou"].value_counts(normalize=True).map("{:.1%}".format))

# se resolvermos isso, já caímos para 18% de cancelamento
# é claro que 100% é utópico, mas com isso já temos as principais causas (ou talvez 3 das principais):
# - forma de contrato mensal
# - necessidade de ligações no call center
# - atraso no pagamento

,idade,sexo,tempo_como_cliente,frequencia_uso,ligacoes_callcenter,dias_atraso,assinatura,duracao_contrato,total_gasto,meses_ultima_interacao,cancelou
0,23.0,Male,13.0,22.0,2.0,1.0,Standard,Annual,909.58,23.0,0.0
2,30.0,Male,7.0,1.0,0.0,8.0,Basic,Annual,768.78,7.0,0.0
3,26.0,Male,40.0,5.0,3.0,8.0,Premium,Annual,398.00,12.0,1.0
6,49.0,Male,6.0,7.0,0.0,0.0,Standard,Annual,751.00,11.0,0.0
7,33.0,Male,15.0,18.0,1.0,20.0,Standard,Annual,839.71,25.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
49991,24.0,Female,33.0,28.0,1.0,11.0,Premium,Annual,844.05,9.0,0.0
49992,37.0,Male,46.0,29.0,2.0,9.0,Standard,Quarterly,517.86,3.0,0.0
49994,63.0,Male,16.0,24.0,2.0,18.0,Standard,Quarterly,442.00,26.0,1.0
49995,62.0,Female,35.0,7.0,2.0,8.0,Basic,Annual,232.00,15.0,1.0


cancelou
0.0    21446
1.0     4821
Name: count, dtype: int64

cancelou
0.0    81.6%
1.0    18.4%
Name: proportion, dtype: object